# MTAM reproduction — ZuCo sentiment analysis

This is the single Colab notebook for the `reproduction` branch of `parmisbathayan/EEG_Language_Alignment`. Data and results remain in Google Drive.

**Current experiment: v17, the frozen canonical sentence-level MLP-EEG baseline.** It uses the exact released 832-feature sentence representation established in v16/v16b and runs one fixed MLP configuration across a predeclared `5 split seeds × 3 model/training seeds = 15 runs`.

The complete distribution is the result. Minimum-validation-loss is the primary checkpoint; maximum-validation-accuracy is saved only as a secondary diagnostic. Test results never select a run, checkpoint, or configuration.

Use a GPU runtime. The driver is resumable: if Colab disconnects, reconnect and run the notebook again. Completed JSONs are validated and skipped.

## 1. Mount Drive and configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OG_ZUCO_SR_DIR = '/content/drive/MyDrive/Thesis/Data/zuco_og_raw'
RESULTS_ROOT = '/content/drive/MyDrive/Thesis/Results/reproduce_EEG_Language_Alignment'
V17_DIR = RESULTS_ROOT + '/v17_MLP_EEG_frozenMultiseed'
EEG_CACHE = V17_DIR + '/eeg_dict_cache_released832_sortedParticipants.pkl'
FORK_URL = 'https://github.com/parmisbathayan/EEG_Language_Alignment.git'
BRANCH = 'reproduction'

import os
assert os.path.isdir(OG_ZUCO_SR_DIR), f'SR .mat folder not found: {OG_ZUCO_SR_DIR}'
mat_files = sorted(name for name in os.listdir(OG_ZUCO_SR_DIR) if name.endswith('.mat'))
assert mat_files, f'No .mat files found in {OG_ZUCO_SR_DIR}'
os.makedirs(V17_DIR, exist_ok=True)
print('Drive mounted. SR files:', mat_files)
print('v17 output:', V17_DIR)

## 2. Clone the current reproduction branch

Re-running this section refreshes the Colab code from GitHub. It does not modify Drive data or completed v17 results.

In [ ]:
%cd /content
!rm -rf /content/EEG_Language_Alignment
!git clone --branch {BRANCH} {FORK_URL}
%cd /content/EEG_Language_Alignment

print('Checked-out commit:')
!git rev-parse HEAD
print('\nChanges relative to upstream:')
!git remote add upstream https://github.com/Jason-Qiu/EEG_Language_Alignment.git 2>/dev/null; git fetch -q upstream
!git log --oneline upstream/main..{BRANCH}

## 3. Verify the Colab runtime

The EEG path imports Transformers even though it does not load BERT. To keep v17 comparable with the earlier MLP runs, this cell installs `transformers==4.40.0` only when the runtime has another version. Colab's existing PyTorch, NumPy, SciPy, pandas, scikit-learn, tqdm and Matplotlib installations are reused.

In [ ]:
import importlib.metadata, subprocess, sys

required_transformers = '4.40.0'
try:
    installed_transformers = importlib.metadata.version('transformers')
except importlib.metadata.PackageNotFoundError:
    installed_transformers = None
if installed_transformers != required_transformers:
    print(f'Installing transformers {required_transformers}; found {installed_transformers}')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        f'transformers=={required_transformers}',
    ])

import numpy, pandas, scipy, sklearn, torch, transformers
assert torch.cuda.is_available(), 'v17 requires a Colab GPU runtime'
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('NumPy:', numpy.__version__, '| SciPy:', scipy.__version__)
print('GPU:', torch.cuda.get_device_name(0))

## 4. Link the original ZuCo data

The `.mat` files are read in place through symlinks. The repository's sentiment-label CSV is copied into the path expected by the released loader. Nothing in the Drive data folder is modified.

In [ ]:
%cd /content/EEG_Language_Alignment
import os, shutil

os.makedirs('data/SR', exist_ok=True)
for folder in ('lr_curves', 'pred_labels', 'baselines'):
    os.makedirs(folder, exist_ok=True)
for filename in mat_files:
    destination = os.path.join('data/SR', filename)
    if not os.path.lexists(destination):
        os.symlink(os.path.join(OG_ZUCO_SR_DIR, filename), destination)
shutil.copy(
    'preprocessed/ZuCo/sentiment_labels_clean.csv',
    'data/sentiment_labels_clean.csv',
)
linked_files = sorted(name for name in os.listdir('data/SR') if name.endswith('.mat'))
assert linked_files == mat_files, (linked_files, mat_files)
labels = pandas.read_csv('data/sentiment_labels_clean.csv')
print('Linked SR files:', linked_files)
print('Labels:', labels.shape, labels.sentiment_label.value_counts().to_dict())

## 5. Frozen v17 design

Every run uses the same setup:

- Exact released sentence features: eight stored `mean_*` bands, first 104 values, participant mean, per-sentence/per-band electrode Z-score, 832 values total.
- Released MLP interpretation: `832 → 256 → 128 → 64 → 3`, four Linear layers total, bias off, ReLU and dropout 0.3. The paper's unresolved ‘six layers’ statement is not guessed into existence.
- Batch 32, cross-entropy, inverse-class-frequency oversampling with replacement, maximum 200 epochs, patience 20 and delta 0.01.
- Released scheduled Adam path: β=`0.9/0.98`, ε=`1e-4`, weight decay `0.01`, warmup 2000. Its schedule overwrites the constructor LR.
- Split seeds `0,1,2,3,4`; model/training seeds `0,1,2`; sampler seed equals model seed.
- Primary checkpoint: minimum validation loss. Secondary diagnostic: maximum validation accuracy, with lower loss as the tie-break.

This is one frozen 15-run experiment—not 15 configurations. Report the mean, variation and range, never only the best run.

## 6. Run or resume all 15 MLP trainings

This may take a while. Each completed run immediately saves a stable JSON and plot in Drive. If Colab disconnects, rerun sections 1–6: the driver validates and skips completed runs, then continues from the first missing run. It refuses to combine results from another commit or configuration.

In [ ]:
import datetime, os, subprocess

driver_timestamp = datetime.datetime.now().strftime('%Y%m%d%H%M%S')
os.makedirs(os.path.join(V17_DIR, 'logs'), exist_ok=True)
driver_log = os.path.join(V17_DIR, 'logs', f'v17_driver_{driver_timestamp}.txt')
driver_command = [
    'python', '-u', 'run_v17_frozen_multiseed.py',
    '--output_dir', V17_DIR,
    '--eeg_cache', EEG_CACHE,
    '--device', 'cuda',
]
print('Running/resuming frozen v17 grid')
print('Output:', V17_DIR)
print('Driver log:', driver_log)
print('-' * 72)
with open(driver_log, 'w') as log_file:
    log_file.write('Command: ' + ' '.join(driver_command) + '\n' + '-' * 72 + '\n')
    process = subprocess.Popen(
        driver_command, cwd='/content/EEG_Language_Alignment',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
        env={**os.environ, 'TQDM_DISABLE': '1'},
    )
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(
        f'v17 stopped with exit code {return_code}. See {driver_log}. '
        'The completed runs remain valid; rerun this cell after resolving the printed error.'
    )
print('-' * 72)
print('v17 complete. Summary:', os.path.join(V17_DIR, 'v17_summary.json'))

## 7. Show the compact result summary

Run after section 6 completes. The summary shows the entire distribution and seed-variance decomposition. Every individual JSON retains exact IDs, hyperparameters, sampling details, learning history, both checkpoint evaluations and labeled confusion matrices.

In [ ]:
import json, os

summary_path = os.path.join(V17_DIR, 'v17_summary.json')
assert os.path.exists(summary_path), 'v17_summary.json is absent; section 6 has not fully completed'
with open(summary_path) as summary_file:
    summary = json.load(summary_file)
assert summary['status'] == 'complete' and summary['completed_runs'] == 15

primary_key = 'primary_minimum_validation_loss'
secondary_key = 'secondary_maximum_validation_accuracy'
for key, label in ((primary_key, 'PRIMARY minimum-loss'), (secondary_key, 'SECONDARY maximum-accuracy')):
    test = summary['checkpoint_summaries'][key]['overall']['test']
    accuracy, f1 = test['accuracy'], test['f1_macro']
    print(f'\n{label} checkpoint across all 15 runs')
    print(
        f"  Accuracy: {accuracy['mean']:.4f} +/- {accuracy['population_standard_deviation']:.4f} "
        f"(range {accuracy['minimum']:.4f}–{accuracy['maximum']:.4f})"
    )
    print(
        f"  Macro F1: {f1['mean']:.4f} +/- {f1['population_standard_deviation']:.4f} "
        f"(range {f1['minimum']:.4f}–{f1['maximum']:.4f})"
    )

print('\nPaper target: accuracy 0.499, macro F1 0.480')
print(
    'Mean always-majority test accuracy:',
    summary['baselines']['accuracy_across_splits']['mean'],
)

print('\nPrimary per-run results')
print(' split model epoch  val_acc  test_acc  test_f1')
for run in summary['per_run']:
    result = run['checkpoints'][primary_key]
    print(
        f" {run['split_seed']:>5} {run['model_seed']:>5} "
        f"{result['checkpoint_epoch']:>5} "
        f"{result['validation_accuracy']:>8.4f} "
        f"{result['test_accuracy']:>9.4f} "
        f"{result['test_f1_macro']:>8.4f}"
    )

print('\nPrimary test variance fractions')
variance = summary['checkpoint_summaries'][primary_key]['variance_decomposition']
for metric in ('test_accuracy', 'test_f1_macro'):
    print(' ', metric, variance[metric]['fractions_of_total'])
print('\nCommit:', summary['code_commit'])
print('Complete summary:', summary_path)

---
### After v17
Tell Codex that v17 completed. We will inspect the Drive summary and all run JSONs, report the primary distribution first, compare it with majority/chance and the paper, then use the predeclared variance decomposition to determine whether split choice or training randomness dominates. We will not promote the best seed or the secondary checkpoint.